In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()
g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

In [ ]:

cycle_features = [
    "soc_start_pct",
    "soc_end_pct",
    "depth_of_discharge_pct",
    "high_soc_exposure_fraction",
    "low_soc_exposure_fraction",
    "charge_c_rate",
    "discharge_c_rate",
    "charge_current_a",
    "discharge_current_a",
    "charge_duration_min",
    "discharge_duration_min",
    "energy_discharged_kwh",
    "ambient_temp_c",
    #"pack_temp_c",
    "cooling_active",
    "pack_voltage_mean_v"
]

extended_features = cycle_features + [
    "battery_age_days",
    "time_since_previous_cycle_hours",
    "equivalent_full_cycles",
    "cumulative_high_temp_hours",
    "pack_voltage_charge_v",
    "pack_voltage_discharge_v"
]

features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
]

features_B = features_A + [
    "internal_resistance_mOhm",
]

features_C = features_B + [
    "cumulative_high_temp_hours",
]


In [ ]:
class_names = ["Normal", "Moderate", "Severe", "Critical"]
class_names_mapping = {class_names[i]:i for i in range(len(class_names))}

def perform_decision_tree(X_train, X_val, y_train, y_val,
                          criterion, max_depth, min_samples_leaf, ccp_alpha):

    tree = DecisionTreeClassifier(
        criterion=criterion,              # "gini" or "entropy"
        max_depth=max_depth,              # None = grow until pure
        min_samples_leaf=min_samples_leaf,
        ccp_alpha=ccp_alpha,              # cost-complexity pruning strength
        class_weight="balanced",
        random_state=42,
    )
    tree.fit(X_train, y_train)
    predicted = tree.predict(X_val)  # is list of strings

    predicted_num = [class_names_mapping[x] for x in predicted]
    y_val_num = [class_names_mapping[x] for x in y_val]

    metrics = {
        "ARI": adjusted_rand_score(y_val_num, predicted_num),
        "NMI": normalized_mutual_info_score(y_val_num, predicted_num),
        "accuracy": accuracy_score(y_val_num, predicted_num),
        "macro_F1": f1_score(y_val, predicted, labels=class_names,
                             average="macro", zero_division=0),
    }

    return metrics, predicted

### ```min_samples_leaf```

In order for the algorithm to make a split, it first determines whether there would be at least this many samples in each child. If this is not the case, the algorithm doesn't split the tree at this branch. Setting a low value of ```min_samples_leaf``` could mean that the model may learn noise, and setting a value too high could mean that the model doesn't split nearly enough to make the tree deep enough.

### ```ccp_alpha```

